In [1]:
import nltk
import numpy as np

text = " "

for i in range(1, 6):

    with open(f"/Users/elmarmammadov/Documents/NLP/Corpora/Fuzuli/fuzuli-{i}.md", "r") as file:
        text += file.read()
        text += " "


In [2]:
from collections import Counter

# Options: "none", "laplace", "k", "interpolation", "backoff", "kneser_ney"
smoothing_mode = "backoff"
tokens = text.split()
freq_dist_temp = nltk.FreqDist(tokens)
rare_words = [w for w in tokens if freq_dist_temp[w] == 1]
tokens = [w if w not in rare_words else "UNK" for w in tokens]
freq_dist = nltk.FreqDist(tokens)
vocab = nltk.FreqDist(tokens).keys()
unigram = np.zeros(len(vocab))
word_to_index = {word: i for i, word in enumerate(vocab)}

bigrams = [(tokens[i], tokens[i + 1]) for i in range(len(tokens)-1)]
bigram_count = Counter(bigrams)
bigram_probabilities = np.zeros([len(vocab), len(vocab)])

trigrams = [(tokens[i], tokens[i + 1], tokens[i + 2]) for i in range(len(tokens)-2)]
trigram_count = Counter(trigrams)
trigram_probabilities = np.zeros([len(list(bigram_count.keys())), len(vocab)])

word_to_index = {word: i for i, word in enumerate(vocab)}
bigram_to_index = {(w1, w2):i for i, (w1,w2) in enumerate(list(bigram_count.keys()))}

In [3]:
def p_unigram(w):
    return freq_dist[w]/len(tokens)

def p_bigram(w1, w2):
    return bigram_count[(w1, w2)]/freq_dist[w1]

def p_trigram(w1, w2, w3):
    return trigram_count[(w1, w2, w3)]/bigram_count[(w1,w2)]

In [4]:
import math

lambda1_bi, lambda2_bi = 0.5, 0.5

for _ in range(20):
    g1 = g2 = 0
    for i in range(1, len(tokens)):
        w1, w2 = tokens[i-1], tokens[i]
        num1 = lambda1_bi * p_unigram(w2)
        num2 = lambda2_bi * p_bigram(w1, w2)
        denom = num1 + num2
        if denom > 0:
            g1 += num1 / denom
            g2 += num2 / denom

    total = g1 + g2
    lambda1_bi = g1 / total
    lambda2_bi = g2 / total

print("Estimated lambdas (bigram):", lambda1_bi, lambda2_bi)

lambda1_tri, lambda2_tri, lambda3_tri = 1/3, 1/3, 1/3

# EM iterations
for _ in range(30):
    g1 = g2 = g3 = 0
    for i in range(2, len(tokens)):
        w1, w2, w3 = tokens[i-2], tokens[i-1], tokens[i]
        num1 = lambda1_tri * p_unigram(w3)
        num2 = lambda2_tri * p_bigram(w2, w3)
        num3 = lambda3_tri * p_trigram(w1, w2, w3)
        denom = num1 + num2 + num3
        if denom > 0:
            g1 += num1 / denom
            g2 += num2 / denom
            g3 += num3 / denom

    total = g1 + g2 + g3
    lambda1_tri = g1 / total
    lambda2_tri = g2 / total
    lambda3_tri = g3 / total

print("Estimated lambdas (trigram):", lambda1_tri, lambda2_tri, lambda3_tri)


def interpolated_bigram_prob(w1, w2):
    return (
        lambda1_bi * p_unigram(w2) +
        lambda2_bi * p_bigram(w1, w2)
    )

def interpolated_trigram_prob(w1, w2, w3):
    return (
        lambda1_tri * p_unigram(w3) +
        lambda2_tri * p_bigram(w2, w3) +
        lambda3_tri * p_trigram(w1, w2, w3)
    )

Estimated lambdas (bigram): 1.1960446176203946e-09 0.9999999988039553
Estimated lambdas (trigram): 1.3962724198379137e-26 3.488751183815916e-16 0.9999999999999997


In [5]:
from collections import defaultdict

D = 0.75  

# Continuation counts: which previous words a word follows
continuation_counts = defaultdict(set)
for (w1, w2) in bigram_count:
    continuation_counts[w2].add(w1)

total_bigram_types = len(bigram_count)

# Count of all bigrams starting with each w1
c_w1_dict = defaultdict(int)
num_bigrams_starting_w1 = defaultdict(int)

for (w1, w2), count in bigram_count.items():
    c_w1_dict[w1] += count
    num_bigrams_starting_w1[w1] += 1

# Precompute continuation probabilities for each w2
continuation_probs = {w2: len(continuation_counts[w2]) / total_bigram_types for w2 in vocab}

trigram_denominator = defaultdict(int)
# Number of unique trigrams starting with (w1, w2) for lambda
num_trigrams_starting_w1_w2 = defaultdict(int)

for (w1, w2, w3), count in trigram_count.items():
    trigram_denominator[(w1, w2)] += count
    num_trigrams_starting_w1_w2[(w1, w2)] += 1

In [6]:
V = len(vocab)
N = len(tokens)

for i, word in enumerate(vocab):

    count = freq_dist[word]

    if smoothing_mode == "none":
        unigram[i] = count / N

    elif smoothing_mode == "laplace":
        unigram[i] = (count + 1) / (N + V)

    elif smoothing_mode == "k":
        unigram[i] = (count + k) / (N + k * V)

print(unigram)

[0. 0. 0. ... 0. 0. 0.]


In [ ]:
lambda_backoff = 0.4

for (w1, w2) in bigrams:
        i = word_to_index[w1]
        j = word_to_index[w2]
        count = bigram_count.get((w1,w2), 0 )
        if smoothing_mode == "none":
            # raw probability
            bigram_probabilities[i, j] = count / freq_dist[w1]

        elif smoothing_mode == "laplace":
            # add-1 smoothing
            bigram_probabilities[i, j] = (count + 1) / (freq_dist[w1] + len(vocab))

        elif smoothing_mode == "k":
            # k-smoothing, e.g., k=0.5
            k = 0.5
            bigram_probabilities[i, j] = (count + k) / (freq_dist[w1] + k*len(vocab))

        elif smoothing_mode == "interpolation":
            # must define lambda1_bi, lambda2_bi and p_unigram, p_bigram
            bigram_probabilities[i, j] = interpolated_bigram_prob(w1, w2)

        elif smoothing_mode == "backoff":
            if (w1, w2) in bigrams:
                bigram_probabilities[i, j] = count / freq_dist[w1]
            else:
                bigram_probabilities[i, j] = lambda_backoff * unigram[j]

        elif smoothing_mode == "kneser_ney":
            c_w1 = c_w1_dict[w1]
            lambda_w1 = D * num_bigrams_starting_w1[w1] / c_w1 if c_w1 > 0 else 1.0
            P_cont = continuation_probs[w2]
            bigram_probabilities[i, j] = max(count - D, 0)/c_w1 + lambda_w1 * P_cont


print(bigram_probabilities)

In [ ]:
lambda_backoff = 0.4
k = 0.5

for (w1, w2, w3) in trigrams:

        # Skip if this bigram row doesn't exist in your matrix
        if (w1, w2) not in bigram_to_index:
            continue

        i = bigram_to_index[(w1, w2)]
        denom = bigram_count.get((w1, w2), 0)



        j = word_to_index[w3]
        count = trigram_count.get((w1, w2, w3), 0)

        if smoothing_mode == "none":
            trigram_probabilities[i, j] = (
                count / denom if denom > 0 else 0)
                

        elif smoothing_mode == "laplace":
            trigram_probabilities[i, j] = (
                    (count + 1) / (denom + len(vocab))
                )

        elif smoothing_mode == "k":
                trigram_probabilities[i, j] = (
                    (count + k) / (denom + k * len(vocab))
                )

        elif smoothing_mode == "interpolation":
                trigram_probabilities[i, j] = (
                    interpolated_trigram_prob(w1, w2, w3)
                )

        elif smoothing_mode == "backoff":

            if count > 0:
                    trigram_probabilities[i, j] = count / denom

            elif bigram_count.get((w2, w3), 0) > 0:
                    trigram_probabilities[i, j] = (
                        lambda_backoff *
                        bigram_probabilities[
                            word_to_index[w2],
                            word_to_index[w3]
                        ]
                    )

            else:
                    trigram_probabilities[i, j] = (
                        lambda_backoff**2 * unigram[j]
                    )

        elif smoothing_mode == "kneser_ney":

            c_w1_w2 = denom

            if c_w1_w2 > 0:
                    lambda_w1_w2 = (
                        D *
                        num_trigrams_starting_w1_w2.get((w1, w2), 0)
                        / c_w1_w2
                    )
            else:
                    lambda_w1_w2 = 1.0

            P_backoff = bigram_probabilities[
                    word_to_index[w2],
                    word_to_index[w3]
                ]

            trigram_probabilities[i, j] = (
                max(count - D, 0) / c_w1_w2
                + lambda_w1_w2 * P_backoff
                if c_w1_w2 > 0 else P_backoff
            )

print(trigram_probabilities)

KeyboardInterrupt: 

In [ ]:
with open("hatai-1.md", "r") as file:
        test = file.read()

test = test.split()
test = [w if w in vocab else "UNK" for w in test]
print(test)

['UNK', 'UNK', '26', '***', 'UNK', 'kim,', 'оldu', 'canü', 'dil', 'heyran', 'UNK', 'UNK', 'UNK', 'dövran', 'UNK', 'Çün', 'sənin', 'hüsnün', 'UNK', 'UNK', 'hər', 'kim', 'ki,', 'görməz,', 'UNK', 'iman', 'UNK', 'UNK', 'ey', 'UNK', 'gözlü', 'UNK', 'Eylə', 'UNK', 'bu', 'bağrım,', 'UNK', 'dərman', 'UNK', 'Gül', 'UNK', 'UNK', 'bəs', 'UNK', 'hər', 'zaman,', 'Qanlı', 'yaşımdan', 'tökər', 'hər', 'dəm', 'gözüm', 'baran', 'UNK', 'UNK', 'könlü', 'bu', 'UNK', 'UNK', 'UNK', 'bənzər', 'kim,', 'görünməz', 'çeşmeyi-heyvan', 'UNK', '***', 'Getdi', 'оl', 'dilbər,', 'bəsi', 'dərdü', 'bəla', 'qaldı', 'mana,', 'Nə', 'bəla,', 'bil', 'kim,', 'UNK', 'cövrü', 'cəfa', 'qaldı', 'mana.', 'UNK', 'gəldim,', 'mən', 'gədayə', 'hiç', 'inayət', 'qılmadın,', 'UNK', 'UNK', 'UNK', 'qaldı', 'mana.', 'Müjdə', 'gəldi', 'UNK', 'ki,', 'qətl', 'оldu', 'rəqib,', 'Şükr', 'kim,', 'biganə', 'getdi,', 'aşina', 'qaldı', 'mana.', 'UNK', 'kövkəb', 'kimi', 'yaş', 'UNK', 'UNK', 'gözlərim,', 'Yer', 'ilə,', 'UNK', 'ilə', 'Keyvan', 'həm', 'ba

In [ ]:
log_prob = 0
N = len(test)

for word in test:
    p = unigram[word_to_index[word]]
    log_prob += np.log(p)

perplexity = np.exp(-log_prob / N)

print(perplexity)

335.4780122413052


In [ ]:
log_prob = 0
bigrams_test = [(test[i], test[i + 1]) for i in range(len(test) - 1)]
bigrams_test = [(w1, w2) if (w1,w2) in bigrams else ("UNK", "UNK") for (w1, w2) in bigrams_test]

for (w1, w2) in bigrams_test:
    p = bigram_probabilities[word_to_index[w1]][word_to_index[w2]]
    log_prob += np.log(p)

perplexity = np.exp(-log_prob / len(bigrams_test))

print(perplexity)

inf


/var/folders/sq/46wrn8lj59s3m4kjsybr6pjw0000gn/T/ipykernel_24484/2261008216.py:7: RuntimeWarning: divide by zero encountered in log
  log_prob += np.log(p)


In [ ]:
log_prob = 0
trigrams_test = [(test[i], test[i + 1], test[i + 2]) for i in range(len(test) -2)]
trigrams_test = [(w1,w2,w3) if (w1,w2,w3) in trigrams else ("UNK", "UNK", "UNK") for (w1,w2,w3) in trigrams_test]

for (w1, w2, w3) in trigrams_test:
    p = trigram_probabilities[bigram_to_index[(w1,w2)] if (w1, w2) in bigram_to_index else bigram_to_index[("UNK", "UNK")]][word_to_index[w3]]
    log_prob += np.log(p)

perplexity = np.exp(-log_prob / len(trigrams_test))

print(perplexity)

inf


/var/folders/sq/46wrn8lj59s3m4kjsybr6pjw0000gn/T/ipykernel_24484/542803205.py:7: RuntimeWarning: divide by zero encountered in log
  log_prob += np.log(p)
